# INFO371 – Application 1: Voice Assistant Review Sentiment Classifier
## Data Loading Notebook

**Application Summary:**  
This application predicts whether a user's written review of a smart home voice assistant device reflects **positive or negative sentiment** (binary classification). The prediction target is the `feedback` column (1 = positive, 0 = negative).

**Dataset:** Amazon Alexa Reviews  
**Source:** Kaggle – [Amazon Alexa Reviews](https://www.kaggle.com/datasets/sid321axn/amazon-alexa-reviews)  



In [ ]:
!pip install pandas --quiet
!pip install kagglehub --quiet

In [ ]:
import kagglehub

path = kagglehub.dataset_download("sid321axn/amazon-alexa-reviews")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'amazon-alexa-reviews' dataset.
Path to dataset files: /kaggle/input/amazon-alexa-reviews


In [ ]:
import pandas as pd
import os

dataset_file_path = os.path.join(path, 'amazon_alexa.tsv')

df = pd.read_csv(dataset_file_path, sep='\t')

print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (3150, 5)


,rating,date,variation,verified_reviews,feedback
0,5,31-Jul-18,Charcoal Fabric,Love my Echo!,1
1,5,31-Jul-18,Charcoal Fabric,Loved it!,1
2,4,31-Jul-18,Walnut Finish,"Sometimes while playing a game, you can answer...",1
3,5,31-Jul-18,Charcoal Fabric,I have had a lot of fun with this thing. My 4 ...,1
4,5,31-Jul-18,Charcoal Fabric,Music,1


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3150 entries, 0 to 3149
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   rating            3150 non-null   int64 
 1   date              3150 non-null   object
 2   variation         3150 non-null   object
 3   verified_reviews  3149 non-null   object
 4   feedback          3150 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 123.2+ KB


In [ ]:
print("Target variable distribution (feedback):")
print(df['feedback'].value_counts())
print(f"\nClass balance: {df['feedback'].mean():.2%} positive")

Target variable distribution (feedback):
feedback
1    2893
0     257
Name: count, dtype: int64

Class balance: 91.84% positive


In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

df = df.dropna(subset=['verified_reviews'])
print(f"\nFinal dataset shape after dropping missing reviews: {df.shape}")

Missing values per column:
rating              0
date                0
variation           0
verified_reviews    1
feedback            0
dtype: int64

Final dataset shape after dropping missing reviews: (3149, 5)


---

The DataFrame `df` is now loaded and ready for modeling. Key columns:

| Column | Role | Description |
|---|---|---|
| `verified_reviews` | Feature | Full text of the user's written review |
| `variation` | Optional Feature | Device model/color variant |
| `rating` | Supplemental | 1–5 star rating (not the target here) |
| `feedback` | **Target** | 1 = positive sentiment, 0 = negative sentiment |

**Your modeling task:** Predict `feedback` (binary) from `verified_reviews` (text).


In [ ]:
df[['verified_reviews', 'variation', 'rating', 'feedback']].head(10)

,verified_reviews,variation,rating,feedback
0,Love my Echo!,Charcoal Fabric,5,1
1,Loved it!,Charcoal Fabric,5,1
2,"Sometimes while playing a game, you can answer...",Walnut Finish,4,1
3,I have had a lot of fun with this thing. My 4 ...,Charcoal Fabric,5,1
4,Music,Charcoal Fabric,5,1
5,I received the echo as a gift. I needed anothe...,Heather Gray Fabric,5,1
6,"Without having a cellphone, I cannot use many ...",Sandstone Fabric,3,1
7,I think this is the 5th one I've purchased. I'...,Charcoal Fabric,5,1
8,looks great,Heather Gray Fabric,5,1
9,Love it! I’ve listened to songs I haven’t hear...,Heather Gray Fabric,5,1


In [ ]:
import plotly.express as px
fig = px.histogram(
    df,
    x="feedback",
    title="Distribution of Alexa Review Sentiment",
    labels={"feedback": "Sentiment (0 = Negative, 1 = Positive)"}
)

fig.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score


# load data
df = pd.read_csv(dataset_file_path, sep='\t')
df = df.dropna(subset=['verified_reviews'])

# features and target
X = df['verified_reviews']
y = df['feedback']

# split data
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 371, stratify = y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size = 0.25, random_state = 371, stratify=y_train_full
)

# convert text to numbers
vectorizer = CountVectorizer(stop_words='english')

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)
X_train_full_vec = vectorizer.fit_transform(X_train_full)
X_test_vec = vectorizer.transform(X_test)

best_model = None
best_recall = 0

# Logistic Regression
for c in [0.1, 1, 10]:
    model = LogisticRegression(C = c)
    model.fit(X_train_vec, y_train)

    y_val_pred = model.predict(X_val_vec)
    rec = recall_score(y_val, y_val_pred, pos_label = 0)

    print("Logistic C =", c, "Recall:", rec)

    if rec > best_recall:
        best_recall = rec
        best_model = ("logistic", c)

# Random Forest
for n in [50, 100, 200]:
    model = RandomForestClassifier(n_estimators=n, random_state=371)
    model.fit(X_train_vec, y_train)

    y_val_pred = model.predict(X_val_vec)
    rec = recall_score(y_val, y_val_pred, pos_label = 0)

    print("Random Forest n =", n, "Recall:", rec)

    if rec > best_recall:
        best_recall = rec
        best_model = ("rf", n)

print("\nBest model:", best_model)

# train final model
if best_model[0] == "logistic":
    final_model = LogisticRegression(C = best_model[1])
else:
    final_model = RandomForestClassifier(n_estimators = best_model[1], random_state = 371)

final_model.fit(X_train_full_vec, y_train_full)

# test evaluation
y_test_pred = final_model.predict(X_test_vec)

print("\nFinal Test Results")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("Precision:", precision_score(y_test, y_test_pred, pos_label=0))
print("Recall:", recall_score(y_test, y_test_pred, pos_label=0))

Logistic C = 0.1 Recall: 0.0784313725490196
Logistic C = 1 Recall: 0.23529411764705882
Logistic C = 10 Recall: 0.35294117647058826
Random Forest n = 50 Recall: 0.21568627450980393
Random Forest n = 100 Recall: 0.21568627450980393
Random Forest n = 200 Recall: 0.21568627450980393

Best model: ('logistic', 10)

Final Test Results
Accuracy: 0.946031746031746
Precision: 0.7575757575757576
Recall: 0.49019607843137253
